# Limpar: Arquivo Morto (Planilha Petrus)

Fonte ID: `00797637-9b3f-447d-acb8-29315c699aa8`

**Workflow:**
1. Carregar dados raw
2. Definir pipeline de limpeza
3. Rodar pipeline
4. Inspecionar resultado
5. Validar contra contrato Silver
6. Exportar ou push para API

In [28]:
import datawork
datawork.setup()

## 1. Carregar Dados Raw

In [29]:
from datawork.io.loaders import load_csv

df_raw = load_csv("../data/arquivo_morto_raw.csv")

print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} cols")
print(f"Colunas: {list(df_raw.columns)}")
df_raw.head(3)

Shape: 1292 rows x 10 cols
Colunas: ['AC', 'AF', 'AT', 'End', 'Local', 'Observ.', 'Proprietário', 'Telefones', 'Valor', 'nº']


,AC,AF,AT,End,Local,Observ.,Proprietário,Telefones,Valor,nº
0,,,,,alpha,,,,,
1,5.500,,5.000,África,alpha,,Rubi's Adilson,7211.2157/4193.4786/4166.5792/,"125.000,00",224
2,3.580,2.611,5.000,África,alpha,4 doca PD 12 18 vagas,JGA Adriane/4 galpões iguais,4195.6974/,"107.000,00",337


## 2. Definir Pipeline

Montar o pipeline de limpeza usando stages composáveis. Ajustar os stages conforme a planilha.

In [30]:
from datawork.pipeline.runner import PipelineRunner
from datawork.pipeline.stages import (
    parse_sections,
    drop_empty_rows,
    detect_units,
    normalize_areas,
    normalize_addresses,
    extract_obs,
    classify_values,
    extract_contacts,
    classify_tipo,
    fill_missing_cidade,
    filter_useless_rows,
    generate_titulo,
    preserve_observacoes,
    geocode_addresses,
    rename_to_silver,
    select_columns,
    compute_dedup_hash,
    drop_duplicates_by_hash,
)

# Configs do formato Petrus
from petrus.infrastructure.mdm.connectors.petrus_config import STREET_CANONICAL, STREET_INFO
from petrus.infrastructure.mdm.connectors.petrus_spreadsheet import SECTION_MARKERS, REGION_CITY_MAP
from pathlib import Path

LEGACY_CACHE = str(Path.home() / "Petros" / "data" / "pipeline" / "output" / "geocache.json")

pipeline = (
    PipelineRunner("arquivo_morto")
    # Estrutura
    .add("sections", lambda df: parse_sections(df, SECTION_MARKERS))
    .add("empty", drop_empty_rows)
    # Pré-processamento
    .add("units", detect_units)                    # AT com códigos de unidade -> coluna "unidade"
    .add("areas", normalize_areas)                  # AT/AC/AF -> float
    # Normalização
    .add("addresses", lambda df: normalize_addresses(df, STREET_CANONICAL, STREET_INFO, REGION_CITY_MAP))
    .add("observations", extract_obs)               # Obs -> 15+ campos estruturados
    .add("values", classify_values)                  # Valor -> locação/venda/m²
    .add("contacts", extract_contacts)               # Proprietário/Telefones -> nome/email/phones
    # Enriquecimento
    .add("tipo", classify_tipo)                      # terreno vs galpão pela região
    .add("fill_cidade", lambda df: fill_missing_cidade(df, default_cidade="Barueri"))
    .add("titulo", generate_titulo)                  # Galpão - Rua X, 123 - Barueri
    .add("observacoes", preserve_observacoes)        # Preserva texto original das Obs
    .add("geocode", lambda df: geocode_addresses(df, seed_cache_path=LEGACY_CACHE))
    # Qualidade
    .add("useless", filter_useless_rows)             # Remove rows sem dados úteis
    .add("dedup", compute_dedup_hash)                # Hash (endereço+área+valor)
    .add("dedup_drop", drop_duplicates_by_hash)      # Remove duplicatas exatas
    # Finalização
    .add("rename", rename_to_silver)                 # pe_direito -> pe_direito_m, etc.
    .add("select", select_columns)                   # Remove colunas raw (AC, AF, End, etc.)
)

## 3. Rodar Pipeline

In [31]:
df_clean = pipeline.run(df_raw)

Pipeline 'arquivo_morto': 18 stages, 1292 rows
--------------------------------------------------
  [sections] 1292 -> 1266 rows (-26), 11 cols (0.08s)
  [empty] 1266 -> 1266 rows, 11 cols (0.00s)
  [units] 1266 -> 1266 rows, 12 cols (0.00s)
  [areas] 1266 -> 1266 rows, 15 cols (0.01s)
  [addresses] 1266 -> 1266 rows, 21 cols (0.19s)
  [observations] 1266 -> 1266 rows, 37 cols (0.11s)
  [values] 1266 -> 1266 rows, 41 cols (0.17s)
  [contacts] 1266 -> 1266 rows, 44 cols (0.15s)
  [tipo] 1266 -> 1266 rows, 45 cols (0.01s)
  [fill_cidade] 1266 -> 1266 rows, 45 cols (0.14s)
  [titulo] 1266 -> 1266 rows, 46 cols (0.03s)
  [observacoes] 1266 -> 1266 rows, 47 cols (0.00s)
  Geocoding: 253 ruas unicas, 253 no cache, 0 para buscar
  Resultado: 806/1266 (63.7%) com coordenadas
  [geocode] 1266 -> 1266 rows, 49 cols (0.14s)
  [useless] 1266 -> 1018 rows (-248), 49 cols (0.06s)
  [dedup] 1018 -> 1018 rows, 50 cols (0.03s)
  [dedup_drop] 1018 -> 965 rows (-53), 50 cols (0.00s)
  [rename] 965 -> 965

In [32]:
pipeline.summary()

,stage,rows_in,rows_out,cols_in,cols_out,elapsed_s
0,sections,1292,1266,10,11,0.077
1,empty,1266,1266,11,11,0.002
2,units,1266,1266,11,12,0.004
3,areas,1266,1266,12,15,0.015
4,addresses,1266,1266,15,21,0.194
5,observations,1266,1266,21,37,0.108
6,values,1266,1266,37,41,0.170
7,contacts,1266,1266,41,44,0.151
8,tipo,1266,1266,44,45,0.007
9,fill_cidade,1266,1266,45,45,0.145


## 4. Inspecionar Resultado

In [33]:
from datawork.display import show_sample, show_stats
from datawork.profiling import completeness_report

show_sample(df_clean, 10, "Amostra dos dados limpos")

print("\n")
completeness_report(df_clean)


  Amostra dos dados limpos
Shape: 965 rows x 40 cols


,regiao,unidade,area_total_m2,area_construida_m2,area_piso_m2,logradouro,numero,complemento,endereco,cidade,bairro,numero_docas,pe_direito_m,vagas_estacionamento,area_escritorio_m2,iptu,valor_condominio,elevador,inquilino,cab_primaria,area_mezanino_m2,status,potencia_eletrica_kva,avcb,gerador,ponte_rolante,zoneamento,valor_locacao,tipo_operacao,valor_venda,preco_m2,proprietario_nome,proprietario_email,proprietario_telefone,tipo,titulo,observacoes,latitude,longitude,hash_dedup
0,Alphaville,None,5000.0,5500.0,NaN,Alameda África,224,,"Alameda África, 224 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125000.0,locacao,NaN,NaN,Rubi's Adilson,None,7211-2157; 4193-4786; 4166-5792,galpao,"Galpão - Alameda África, 224 - Barueri",None,None,None,467f8716d99f06b9
1,Alphaville,None,5000.0,3580.0,2611.0,Alameda África,337,,"Alameda África, 337 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,4.0,12.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,107000.0,locacao,NaN,NaN,JGA Adriane/4 galpões iguais,None,4195-6974,galpao,"Galpão - Alameda África, 337 - Barueri",4 doca PD 12 18 vagas,None,None,3c943f3288091a8e
2,Alphaville,None,5000.0,3580.0,2611.0,Alameda África,337,,"Alameda África, 337 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,4.0,12.0,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93000.0,locacao,NaN,NaN,JGA Adriane,None,4195-6974,galpao,"Galpão - Alameda África, 337 - Barueri",4 doca PD12 18 vaga,None,None,3353cd3c6917147b
3,Alphaville,None,NaN,4607.0,3900.0,Alameda África,545,,"Alameda África, 545 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,3.0,10.0,40.0,770.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125000.0,locacao,NaN,NaN,RUBI´S,None,94448-1467,galpao,"Galpão - Alameda África, 545 - Barueri",770 escr 3doca 40 vagas 10PD,None,None,d1b318f2f9592f52
4,Alphaville,None,2500.0,2238.0,1422.0,Alameda África,685,,"Alameda África, 685 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,NaN,11.0,NaN,NaN,1400.0,500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60500.0,locacao,NaN,NaN,None,clobo.coque@uol.com.br,4326-4761; 99500-9775; 97522-5848,galpao,"Galpão - Alameda África, 685 - Barueri",11PD IPTU 1.400 Cond 500.,None,None,c66e29d0cb6fb726
5,Alphaville,None,5320.0,4400.0,4200.0,Alameda África,734,,"Alameda África, 734 - Alphaville Industrial, B...",Barueri,Alphaville Industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110000.0,locacao,NaN,NaN,Fábio,None,4071-2355; 8217-4000,galpao,"Galpão - Alameda África, 734 - Barueri",None,None,None,484185068287fd67
6,Alphaville,None,11938.0,5618.0,NaN,Alameda Aldeinha,181,,"Alameda Aldeinha, 181 - Alphaville Industrial,...",Barueri,Alphaville Industrial,NaN,7.0,200.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Cesar@indusvest.com.br,3062-3200,galpao,"Galpão - Alameda Aldeinha, 181 - Barueri",7PD 200 vaga,None,None,f4ac621798bcff96
7,Alphaville,None,NaN,NaN,NaN,Alameda Aldeinha,220,,"Alameda Aldeinha, 220 - Alphaville Industrial,...",Barueri,Alphaville Industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,90000.0,ambos,18000000.0,NaN,Luiz Carlos Corretor,None,94223-5333,galpao,"Galpão - Alameda Aldeinha, 220 - Barueri",18milhões,None,None,aae703bf2da3a93f
8,Alphaville,None,9600.0,6700.0,NaN,Alameda Aldeinha,251,,"Alameda Aldeinha, 251 - Alphaville Industrial,...",Barueri,Alphaville Industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fajor empr Imob Alessandro,None,3176-1010,galpao,"Galpão - Alameda Aldeinha, 251 - Barueri",None,None,None,600ae481fbe4ed58
9,Alphaville,None,3750.0,3300.0,NaN,Alameda Aldeinha,252,,"Alameda Aldeinha, 252 - Alphaville Industrial,...",Barueri,Alphaville Industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80000.0,ambos,20000000.0,NaN,None,diogomadv01@gmail.com,98222-1818; 94223-5333,galpao,

,column,non_null,non_empty,pct_filled
0,regiao,965,965,100.0
1,logradouro,965,965,100.0
2,cidade,965,965,100.0
3,titulo,965,965,100.0
4,tipo,965,965,100.0
5,hash_dedup,965,965,100.0
6,endereco,965,965,100.0
7,proprietario_telefone,894,894,92.6
8,numero,965,814,84.4
9,tipo_operacao,773,773,80.1


In [34]:
show_stats(df_clean)

# Verificar distribuição das novas colunas
print("\n=== Distribuição tipo ===")
print(df_clean["tipo"].value_counts().to_string())

print("\n=== Distribuição tipo_operacao ===")
if "tipo_operacao" in df_clean.columns:
    print(df_clean["tipo_operacao"].value_counts(dropna=False).to_string())

print("\n=== Distribuição cidade ===")
if "cidade" in df_clean.columns:
    print(df_clean["cidade"].value_counts(dropna=False).head(15).to_string())

print("\n=== Titulo (amostra) ===")
if "titulo" in df_clean.columns:
    filled = df_clean["titulo"].notna().sum()
    print(f"  Preenchidos: {filled}/{len(df_clean)} ({filled/len(df_clean)*100:.1f}%)")
    print(df_clean["titulo"].dropna().head(5).to_string())
else:
    print("  FALTANDO - coluna titulo nao existe")

print("\n=== Geocoding ===")
if "latitude" in df_clean.columns:
    geo = df_clean["latitude"].notna().sum()
    print(f"  Com coordenadas: {geo}/{len(df_clean)} ({geo/len(df_clean)*100:.1f}%)")
    no_geo = df_clean[df_clean["latitude"].isna()]
    if len(no_geo) > 0:
        ruas_sem = no_geo["logradouro"].value_counts().head(10)
        print(f"  Ruas sem geocoding ({len(no_geo)} imoveis):")
        for rua, count in ruas_sem.items():
            print(f"    {rua}: {count}")
else:
    print("  FALTANDO - coluna latitude nao existe")

print("\n=== Contatos ===")
for col in ["proprietario_nome", "proprietario_email", "proprietario_telefone"]:
    if col in df_clean.columns:
        filled = df_clean[col].fillna("").str.strip().ne("").sum()
        print(f"  {col}: {filled}/{len(df_clean)} ({filled/len(df_clean)*100:.1f}%)")

print("\n=== Colunas no output ===")
print(f"  Total: {len(df_clean.columns)}")
raw_still = [c for c in ["AC","AF","AT","End","Local","Observ.","Proprietário","Telefones","Valor","nº"] if c in df_clean.columns]
if raw_still:
    print(f"  Raw ainda presentes: {raw_still}")
else:
    print(f"  Raw removidas: OK")


Total registros: 965

Campos numéricos:
       area_total_m2  area_construida_m2  area_piso_m2  valor_locacao   valor_venda  pe_direito_m
count     574.000000          756.000000    407.000000     711.000000  7.400000e+01    331.000000
mean     6739.777003         3364.580688   2411.747297   70939.994374  1.511928e+07      9.494381
min       120.000000          100.000000     14.150000    1800.000000  1.200000e+06      2.700000
max    112855.000000        62362.000000  29870.000000  913711.000000  7.700000e+07     16.000000

cidade:
  Barueri: 808
  Santana de Parnaíba: 81
  Jandira: 43
  Araçariguama: 13
  Carapicuíba: 7
  Osasco: 6
  Cotia: 5
  Itapevi: 1
  São Roque: 1

tipo_operacao:
  locacao: 699
  nan: 192
  venda: 62
  ambos: 12

tipo:
  galpao: 883
  terreno: 82

status:
  nan: 959
  vago: 2
  vendido: 2
  obra: 1
  reformado: 1

regiao:
  Alphaville: 467
  Barueri: 265
  Terrenos: 82
  Santana de Parnaíba: 43
  Polo Industrial Jandira: 42
  Fazendinha: 37
  Araçariguama: 9
 

### 4b. Verificação de Resultados Absurdos

Checklist de sanidade para garantir que o pipeline não produziu valores impossíveis.

In [35]:
import numpy as np

print("=" * 60)
print("VERIFICACAO DE RESULTADOS ABSURDOS")
print("=" * 60)

issues = []

# 1. Locacao < R$500
if "valor_locacao" in df_clean.columns:
    low_rent = df_clean[df_clean["valor_locacao"] < 500]
    if len(low_rent) > 0:
        issues.append(f"ALERTA: {len(low_rent)} locacoes < R$500/mes")
        print(f"\n--- Locacoes < R$500 ({len(low_rent)} rows) ---")
        for _, r in low_rent.iterrows():
            print(f"  {r.get('logradouro', '?')} {r.get('numero', '')} | R${r['valor_locacao']:,.0f}")
    else:
        print("\n[OK] Nenhuma locacao < R$500")

# 2. Locacao > R$500k
if "valor_locacao" in df_clean.columns:
    high_rent = df_clean[df_clean["valor_locacao"] > 500_000]
    if len(high_rent) > 0:
        print(f"\n[INFO] {len(high_rent)} locacoes > R$500k (galp. grandes, verificados OK)")
    else:
        print("[OK] Nenhuma locacao > R$500k")

# 3. Venda < R$500k
if "valor_venda" in df_clean.columns:
    low_sale = df_clean[df_clean["valor_venda"] < 500_000]
    if len(low_sale) > 0:
        issues.append(f"ALERTA: {len(low_sale)} vendas < R$500k")
        print(f"\n--- Vendas < R$500k ({len(low_sale)} rows) ---")
        for _, r in low_sale.iterrows():
            print(f"  {r.get('logradouro', '?')} {r.get('numero', '')} | R${r['valor_venda']:,.0f}")
    else:
        print("[OK] Nenhuma venda < R$500k")

# 4. AC > 1.5x AT
if "area_construida_m2" in df_clean.columns and "area_total_m2" in df_clean.columns:
    both = df_clean[df_clean["area_construida_m2"].notna() & df_clean["area_total_m2"].notna()]
    impossible = both[both["area_construida_m2"] > both["area_total_m2"] * 1.5]
    if len(impossible) > 0:
        print(f"\n[INFO] {len(impossible)} com AC > 1.5x AT (multi-andar ou dados da planilha)")

# 5. Pe direito fora 3-20m
if "pe_direito_m" in df_clean.columns:
    pd_vals = df_clean["pe_direito_m"].dropna()
    weird_pd = df_clean[(df_clean["pe_direito_m"] < 3) | (df_clean["pe_direito_m"] > 20)]
    if len(weird_pd) > 0:
        issues.append(f"ALERTA: {len(weird_pd)} com pe direito fora de 3-20m")
        print(f"\n--- Pe direito fora 3-20m ({len(weird_pd)} rows) ---")
        for _, r in weird_pd.head(5).iterrows():
            print(f"  {r.get('logradouro', '?')} {r.get('numero', '')} | PD={r['pe_direito_m']}m")
    else:
        print(f"[OK] {len(pd_vals)} pes direitos entre 3-20m")

# 6. Cidade vazia
if "cidade" in df_clean.columns:
    empty_city = df_clean[df_clean["cidade"].fillna("").str.strip() == ""]
    if len(empty_city) > 0:
        issues.append(f"ALERTA: {len(empty_city)} sem cidade")
    else:
        print("[OK] Todas as linhas tem cidade")

# 7. VENDIDO com status correto
if "status" in df_clean.columns:
    vendido = df_clean[df_clean["status"] == "vendido"]
    print(f"[OK] {len(vendido)} com status='vendido'")

# 8. Nomes que sao URLs
if "proprietario_nome" in df_clean.columns:
    url_names = df_clean[df_clean["proprietario_nome"].fillna("").str.contains(r"www\.|\.com", case=False, na=False)]
    if len(url_names) > 0:
        issues.append(f"ALERTA: {len(url_names)} nomes que sao URLs")
        for _, r in url_names.head(3).iterrows():
            print(f"  nome=\"{r['proprietario_nome']}\"")
    else:
        print("[OK] Nenhum nome e URL")

# 9. Telefones com DDD impossivel
if "proprietario_telefone" in df_clean.columns:
    bad_ddd = df_clean[df_clean["proprietario_telefone"].fillna("").str.contains(r"\((?:20|[3-9]0)\)", na=False)]
    if len(bad_ddd) > 0:
        issues.append(f"ALERTA: {len(bad_ddd)} telefones com DDD impossivel")
        for _, r in bad_ddd.head(3).iterrows():
            print(f"  tel=\"{r['proprietario_telefone']}\"")
    else:
        print("[OK] Nenhum telefone com DDD impossivel")

# Resumo
print("\n" + "=" * 60)
if issues:
    print(f"RESULTADO: {len(issues)} alertas")
    for i in issues:
        print(f"  - {i}")
else:
    print("RESULTADO: Nenhum valor absurdo encontrado!")

VERIFICACAO DE RESULTADOS ABSURDOS

[OK] Nenhuma locacao < R$500

[INFO] 10 locacoes > R$500k (galp. grandes, verificados OK)
[OK] Nenhuma venda < R$500k

[INFO] 16 com AC > 1.5x AT (multi-andar ou dados da planilha)

--- Pe direito fora 3-20m (1 rows) ---
  Alameda Juruá 253 | PD=2.7m
[OK] Todas as linhas tem cidade
[OK] 2 com status='vendido'
[OK] Nenhum nome e URL
[OK] Nenhum telefone com DDD impossivel

RESULTADO: 1 alertas
  - ALERTA: 1 com pe direito fora de 3-20m


## 5. Validar contra Contrato Silver

In [36]:
from datawork.contracts.silver import CleanRecordSchema

try:
    df_validated = CleanRecordSchema.validate(df_clean, lazy=True)
    print("Validação Silver OK!")
except Exception as e:
    print("Erros de validação:")
    print(e)

Validação Silver OK!


## 6. Comparar com Legacy ETL

Comparar nosso resultado com o output do pipeline anterior (`data/pipeline/output/imoveis_limpos.csv`).

In [37]:
import pandas as pd
from pathlib import Path

# O legacy ETL fica fora do repo petrusweb
legacy_path = Path.home() / "Petros" / "data" / "pipeline" / "output" / "imoveis_limpos.csv"
try:
    df_legacy = pd.read_csv(legacy_path, encoding="utf-8-sig")
    print(f"Legacy ETL: {len(df_legacy)} rows, {len(df_legacy.columns)} cols")
    print(f"Novo pipeline: {len(df_clean)} rows, {len(df_clean.columns)} cols")
    print(f"Delta rows: {len(df_clean) - len(df_legacy):+d}")

    # Comparar campos em comum
    legacy_has_area = df_legacy["area_construida"].notna().sum()
    new_has_area = df_clean["area_construida_m2"].notna().sum() if "area_construida_m2" in df_clean.columns else 0
    legacy_has_valor = (df_legacy["valor_aluguel"].notna().sum() + df_legacy["valor_venda"].notna().sum())
    new_has_valor = 0
    if "valor_locacao" in df_clean.columns:
        new_has_valor += df_clean["valor_locacao"].notna().sum()
    if "valor_venda" in df_clean.columns:
        new_has_valor += df_clean["valor_venda"].notna().sum()

    # Campos de obs
    legacy_has_pd = df_legacy["pe_direito"].notna().sum() if "pe_direito" in df_legacy.columns else 0
    new_has_pd = df_clean["pe_direito_m"].notna().sum() if "pe_direito_m" in df_clean.columns else 0
    legacy_has_docas = df_legacy["docas"].notna().sum() if "docas" in df_legacy.columns else 0
    new_has_docas = df_clean["numero_docas"].notna().sum() if "numero_docas" in df_clean.columns else 0

    print(f"\nCampos preenchidos (legacy vs novo):")
    print(f"  area_construida: {legacy_has_area} vs {new_has_area}")
    print(f"  valor (aluguel+venda): {legacy_has_valor} vs {new_has_valor}")
    print(f"  pe_direito: {legacy_has_pd} vs {new_has_pd}")
    print(f"  docas: {legacy_has_docas} vs {new_has_docas}")
    print(f"  colunas: {len(df_legacy.columns)} vs {len(df_clean.columns)} (sem raw)")
except FileNotFoundError:
    print(f"Arquivo do legacy ETL nao encontrado em: {legacy_path}")

Legacy ETL: 1266 rows, 43 cols
Novo pipeline: 965 rows, 40 cols
Delta rows: -301

Campos preenchidos (legacy vs novo):
  area_construida: 803 vs 756
  valor (aluguel+venda): 823 vs 785
  pe_direito: 354 vs 331
  docas: 226 vs 205
  colunas: 43 vs 40 (sem raw)


In [38]:
## 7. Exportar ou Push para API

In [39]:
from datawork.io.pushers import export_csv, push_clean_to_api

# Salvar CSV limpo
export_csv(df_clean, "../output/arquivo_morto_limpo.csv")

# Push para API + gerar cards (descomentar quando pronto)
# Fluxo: insere clean records no DB -> gera cards automaticamente
result = push_clean_to_api(df_clean, fonte_id="00797637-9b3f-447d-acb8-29315c699aa8")

Exported 965 rows to ..\output\arquivo_morto_limpo.csv
Pushing 965 clean records for fonte 00797637-9b3f-447d-acb8-29315c699aa8...


ConnectError: [WinError 10061] Nenhuma conexão pôde ser feita porque a máquina de destino as recusou ativamente